# 02 · Exploratory Analysis and Model Training

This notebook loads the cleaned dataset, examines churn patterns, selects among three classifiers using a validation split from the training partition, evaluates the selected and comparison models on the untouched held-out test set, and saves one selected model bundle for the dashboard.

## 1. Load data and define the split

The split and model settings match the existing project: stratified 80/20 train-test split, random state 42, and a 25% validation slice from the training data. `customerID` and the target are excluded from the predictors.

In [1]:
from pathlib import Path
import joblib
import pandas as pd
import plotly.express as px
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'Datasets').exists():
    PROJECT_ROOT = PROJECT_ROOT / 'Customer_Churn_Prediction'
DATA_PATH = PROJECT_ROOT / 'Datasets' / 'Telco-Customer-Churn_Cleaned.csv'
MODEL_PATH = PROJECT_ROOT / 'selected_model.joblib'
df = pd.read_csv(DATA_PATH)
df['SeniorCitizen'] = df['SeniorCitizen'].astype('category')
feature_columns = [column for column in df.columns if column not in ['customerID', 'Churn']]
X = df[feature_columns]
y = df['Churn'].map({'No': 0, 'Yes': 1})
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)
X_fit, X_validation, y_fit, y_validation = train_test_split(X_train, y_train, test_size=0.25, random_state=42, stratify=y_train)
numeric_columns = X_fit.select_dtypes(include=['number']).columns.tolist()
categorical_columns = [column for column in feature_columns if column not in numeric_columns]
print('Rows:', len(df), '| Training:', len(X_train), '| Validation:', len(X_validation), '| Test:', len(X_test))
print('Churn share:', f'{y.mean():.1%}')

Rows: 7043 | Training: 5634 | Validation: 1409 | Test: 1409
Churn share: 26.5%


## 2. Exploratory data analysis

Charts show the class balance and sample churn rates by contract, tenure group, payment method, and internet service. These are descriptive comparisons and do not imply causation.

In [2]:
churn_counts = df['Churn'].value_counts().reindex(['No', 'Yes']).rename_axis('Churn').reset_index(name='Customers')
fig = px.pie(churn_counts, values='Customers', names='Churn', hole=0.58, color='Churn', color_discrete_map={'No': '#2F6B8A', 'Yes': '#E07A5F'}, title='Customer churn distribution')
fig.show()

for column in ['Contract', 'PaymentMethod', 'InternetService']:
    rates = df.groupby(column, observed=True)['Churn'].apply(lambda values: values.eq('Yes').mean()).rename('Churn rate').reset_index()
    fig = px.bar(rates, x=column, y='Churn rate', color='Churn rate', color_continuous_scale=['#a9c7d6', '#e07a5f'], title=f'Churn rate by {column}')
    fig.update_layout(yaxis_tickformat='.0%', coloraxis_showscale=False)
    fig.show()

tenure_groups = df.assign(TenureGroup=pd.cut(df['tenure'], bins=[-1, 6, 12, 24, 48, 72], labels=['0–6', '7–12', '13–24', '25–48', '49–72']))
tenure_rates = tenure_groups.groupby('TenureGroup', observed=True)['Churn'].apply(lambda values: values.eq('Yes').mean()).rename('Churn rate').reset_index()
fig = px.bar(tenure_rates, x='TenureGroup', y='Churn rate', color='Churn rate', color_continuous_scale=['#a9c7d6', '#e07a5f'], title='Churn rate by tenure group')
fig.update_layout(yaxis_tickformat='.0%', coloraxis_showscale=False)
fig.show()
fig = px.box(df, x='Churn', y='MonthlyCharges', color='Churn', color_discrete_map={'No': '#2F6B8A', 'Yes': '#E07A5F'}, title='Monthly charges by churn status')
fig.show()

## 3. Train and select using validation data

Each model uses the same imputation and categorical encoding. Numeric scaling is applied only to Logistic Regression. Balanced class weights address the target imbalance. Validation F1 is the selection criterion; recall and ROC-AUC break ties. The test split is not used for selection.

In [3]:
def make_pipeline(classifier, scale_numeric=False):
    numeric_steps = [('imputer', SimpleImputer(strategy='median'))]
    if scale_numeric:
        numeric_steps.append(('scaler', StandardScaler()))
    numeric_pipeline = Pipeline(numeric_steps)
    categorical_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OneHotEncoder(handle_unknown='ignore')),
    ])
    preprocessor = ColumnTransformer([
        ('numeric', numeric_pipeline, numeric_columns),
        ('categorical', categorical_pipeline, categorical_columns),
    ])
    return Pipeline([('preprocessor', preprocessor), ('classifier', classifier)])

model_settings = {
    'Logistic Regression': (LogisticRegression(max_iter=2000, class_weight='balanced', random_state=42), True),
    'Decision Tree': (DecisionTreeClassifier(max_depth=6, min_samples_leaf=20, class_weight='balanced', random_state=42), False),
    'Random Forest': (RandomForestClassifier(n_estimators=300, min_samples_leaf=5, class_weight='balanced', random_state=42, n_jobs=-1), False),
}
validation_rows = []
for name, (classifier, scale_numeric) in model_settings.items():
    model = make_pipeline(classifier, scale_numeric)
    model.fit(X_fit, y_fit)
    validation_class = model.predict(X_validation)
    validation_probability = model.predict_proba(X_validation)[:, 1]
    validation_rows.append({
        'Model': name,
        'F1-score': f1_score(y_validation, validation_class, zero_division=0),
        'Recall': recall_score(y_validation, validation_class, zero_division=0),
        'ROC-AUC': roc_auc_score(y_validation, validation_probability),
    })
validation_results = pd.DataFrame(validation_rows).sort_values(['F1-score', 'Recall', 'ROC-AUC'], ascending=False).reset_index(drop=True)
display(validation_results.round(3))
selected_model_name = validation_results.loc[0, 'Model']
print('Selected by validation:', selected_model_name)

,Model,F1-score,Recall,ROC-AUC
0,Random Forest,0.625,0.727,0.834
1,Logistic Regression,0.624,0.789,0.836
2,Decision Tree,0.603,0.816,0.823


Selected by validation: Random Forest


## 4. Held-out test comparison and saved model

Each candidate is refitted on the full training partition and evaluated once on the held-out test rows. The selected model bundle contains the fitted pipeline plus the test risk table and result summaries needed by the dashboard, avoiding duplicate output files.

In [4]:
test_rows = []
fitted_models = {}
test_predictions = {}
for name, (classifier, scale_numeric) in model_settings.items():
    model = make_pipeline(classifier, scale_numeric)
    model.fit(X_train, y_train)
    predicted = model.predict(X_test)
    probability = model.predict_proba(X_test)[:, 1]
    fitted_models[name] = model
    test_predictions[name] = (predicted, probability)
    test_rows.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, predicted),
        'Precision': precision_score(y_test, predicted, zero_division=0),
        'Recall': recall_score(y_test, predicted, zero_division=0),
        'F1-score': f1_score(y_test, predicted, zero_division=0),
        'ROC-AUC': roc_auc_score(y_test, probability),
    })
test_results = pd.DataFrame(test_rows).sort_values(['F1-score', 'Recall', 'ROC-AUC'], ascending=False).reset_index(drop=True)
display(test_results.style.format({column: '{:.3f}' for column in test_results.columns if column != 'Model'}))
fig = px.bar(test_results.melt(id_vars='Model', var_name='Metric', value_name='Score'), x='Metric', y='Score', color='Model', barmode='group', color_discrete_sequence=['#2F6B8A', '#E07A5F', '#49A078'], title='Held-out model comparison')
fig.update_layout(yaxis_range=[0, 1], yaxis_tickformat='.0%')
fig.show()

selected_model = fitted_models[selected_model_name]
selected_class, selected_probability = test_predictions[selected_model_name]
importance_values = selected_model.named_steps['classifier'].feature_importances_
encoded_names = selected_model.named_steps['preprocessor'].get_feature_names_out()
importance_groups = []
for encoded_name, importance_value in zip(encoded_names, importance_values):
    feature_name = encoded_name.split('__', 1)[-1]
    feature_group = next((column for column in feature_columns if feature_name == column or feature_name.startswith(column + '_')), feature_name)
    importance_groups.append({'feature_group': feature_group, 'importance': importance_value})
feature_importance = pd.DataFrame(importance_groups).groupby('feature_group', as_index=False)['importance'].sum().sort_values('importance', ascending=False).reset_index(drop=True)
display(feature_importance.head(12))
importance_chart = px.bar(feature_importance.head(12).sort_values('importance'), x='importance', y='feature_group', orientation='h', color='importance', color_continuous_scale='Teal', title='Most influential feature groups')
importance_chart.update_layout(coloraxis_showscale=False, xaxis_title='Relative importance', yaxis_title='')
importance_chart.show()

confusion = confusion_matrix(y_test, selected_class, labels=[0, 1])
display(pd.DataFrame(confusion, index=['Actual retained', 'Actual churned'], columns=['Predicted retained', 'Predicted churned']))
risk_data = df.loc[X_test.index, ['customerID', 'Contract', 'PaymentMethod', 'tenure', 'MonthlyCharges', 'Churn']].copy()
risk_data['actual_churn'] = y_test.values
risk_data['predicted_churn'] = selected_class
risk_data['churn_probability'] = selected_probability
risk_data['risk_band'] = pd.cut(risk_data['churn_probability'], bins=[-0.001, 0.35, 0.65, 1.001], labels=['Lower', 'Medium', 'Higher'])
risk_data = risk_data.sort_values('churn_probability', ascending=False).reset_index(drop=True)
test_prediction_table = pd.DataFrame({'actual': y_test.values, 'predicted': selected_class, 'probability': selected_probability})
model_bundle = {
    'pipeline': selected_model,
    'features': feature_columns,
    'model_name': selected_model_name,
    'test_metrics': test_results,
    'validation_metrics': validation_results,
    'feature_importance': feature_importance,
    'test_risk': risk_data,
    'test_predictions': test_prediction_table,
}
joblib.dump(model_bundle, MODEL_PATH)
loaded_bundle = joblib.load(MODEL_PATH)
sample_customer = X_test.iloc[[0]]
print('Saved selected model:', MODEL_PATH)
print('Selected model:', loaded_bundle['model_name'])
print('Sample estimated churn probability:', round(float(loaded_bundle['pipeline'].predict_proba(sample_customer)[0, 1]), 3))
print('Model comparison and risk table are stored inside the selected model bundle.')

,Model,Accuracy,Precision,Recall,F1-score,ROC-AUC
0,Random Forest,0.773,0.553,0.751,0.637,0.842
1,Decision Tree,0.740,0.507,0.794,0.619,0.834
2,Logistic Regression,0.738,0.504,0.783,0.614,0.842


,feature_group,importance
0,Contract,0.193335
1,tenure,0.129166
2,TotalCharges,0.104750
3,OnlineSecurity,0.082980
4,MonthlyCharges,0.074372
5,InternetService,0.069413
6,TechSupport,0.067506
7,PaymentMethod,0.057422
8,OnlineBackup,0.036060
9,DeviceProtection,0.027305


,Predicted retained,Predicted churned
Actual retained,808,227
Actual churned,93,281


Saved selected model: D:\Documents\ChatGPT\predictive\Customer_Churn_Prediction\selected_model.joblib
Selected model: Random Forest
Sample estimated churn probability: 0.02
Model comparison and risk table are stored inside the selected model bundle.


## 5. Interpretation

The Random Forest's feature importance is aggregated back to original fields after one-hot encoding. Importance describes model reliance and is not causation. The churn probability is a classification estimate from historical attributes, not an estimate of when a customer will leave. The selected model and its actual results are shared with `app.py` through one saved bundle.